In [260]:
from SPARQLWrapper import SPARQLWrapper, JSON
import pandas as pd

In [261]:
#carichiamo il csv con i soggetti di Wikidata
df_wikidata = pd.read_csv("wd_dataset.csv", encoding="utf-8")

df_wikidata.head(3)


,entity,label,aliases,description,category
0,http://www.wikidata.org/entity/Q304227,Aba,NaN,ninfa naiade della mitologia greca,greek_deity
1,http://www.wikidata.org/entity/Q279782,Abarbarea,NaN,naiade della mitologia greca,greek_deity
2,http://www.wikidata.org/entity/Q2370052,Acanto,NaN,"personaggio della mitologia greca, probabilmente una ninfa amata da Apollo",greek_deity


In [262]:
#estraggo tutte le label  
labels = (
    df_wikidata["label"]
    .dropna() #tolgo i valori nulli
    .str.strip() #tolgo gli spazi bianchi all'inizio e alla fine
    .tolist() #converto in lista

)

#estraggo gli aliases separando quelli multipli
aliases = []
for alias_str in df_wikidata["aliases"].dropna(): #per ogni stringa di alias che non è nulla
    alias_list = [alias.strip() for alias in alias_str.split(",")] #separo gli alias con la virgola, tolgo gli spazi e metto tutto in minuscolo
    aliases.extend(alias_list) #aggiungo alla lista totale degli alias

#unisco tutto 
myth_terms = list(set(labels + aliases)) #unisco le label e gli alias, tolgo i duplicati con set e riconverto in lista
len(myth_terms)

3206

In [263]:
#ENDPOINT SPARLQ DI ARCO 
arco_endpoint = "https://dati.cultura.gov.it/sparql"

sparql_arco = SPARQLWrapper(arco_endpoint)
sparql_arco.setReturnFormat(JSON)

In [264]:
# NUMERO TOTALE DEI BENI STORICO ARTISTICI - senza duplicati 
query1 = """
PREFIX arco: <https://w3id.org/arco/ontology/arco/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#> 

SELECT (COUNT(DISTINCT ?item) AS ?totalItems)
WHERE {
  ?item rdf:type <https://w3id.org/arco/ontology/arco/HistoricOrArtisticProperty> .
}
"""


sparql_arco.setQuery(query1)
results = sparql_arco.query().convert()

for result in results["results"]["bindings"]:
    print(f"Total Historic or Artistic Properties: {result['totalItems']['value']}")

Total Historic or Artistic Properties: 2258640


In [265]:
#query pr estrarre i dati dei beni storico artistici in batch 
# Iterare su tutti i 2.258.640 beni con batch di 20000

rows = []
batch_size = 20000
total_items = 2258640

for offset in range(0, total_items, batch_size):
    print(f"Processando batch: offset={offset}, items processati={len(rows)}")
    
    query2 = f"""
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX a-cd: <https://w3id.org/arco/ontology/context-description/>

SELECT ?item ?subject ?label
WHERE {{

  ?item rdf:type <https://w3id.org/arco/ontology/arco/HistoricOrArtisticProperty> .
  
  OPTIONAL {{ ?item rdfs:label ?label . }}

  OPTIONAL {{ ?item a-cd:subject ?subject . }}

}}
LIMIT {batch_size}
OFFSET {offset}
"""
    
    sparql_arco.setQuery(query2)
    results = sparql_arco.query().convert()
    
    # estrai i dati da questo batch
    for r in results["results"]["bindings"]:
        rows.append({
            "item": r["item"]["value"] if "item" in r else None,
            "label": r["label"]["value"] if "label" in r else None,
            "subject": r["subject"]["value"] if "subject" in r else None
        })
    
    # se il batch è vuoto, ferma il loop
    if len(results["results"]["bindings"]) < batch_size:
        print(f"Batch incompleto ({len(results['results']['bindings'])} items), fine loop")
        break


Processando batch: offset=0, items processati=0
Processando batch: offset=20000, items processati=20000
Processando batch: offset=40000, items processati=40000
Processando batch: offset=60000, items processati=60000
Processando batch: offset=80000, items processati=80000
Processando batch: offset=100000, items processati=100000
Processando batch: offset=120000, items processati=120000
Processando batch: offset=140000, items processati=140000
Processando batch: offset=160000, items processati=160000
Processando batch: offset=180000, items processati=180000
Processando batch: offset=200000, items processati=200000
Processando batch: offset=220000, items processati=220000
Processando batch: offset=240000, items processati=240000
Processando batch: offset=260000, items processati=260000
Processando batch: offset=280000, items processati=280000
Processando batch: offset=300000, items processati=300000
Processando batch: offset=320000, items processati=320000
Processando batch: offset=340000

In [268]:
# i rows sono già stati accumulati dal loop nel batch precedente
# creiamo il dataframe consolidato
df_arco = pd.DataFrame(rows)
df_arco = df_arco.drop_duplicates(subset="item")

print(f"Righe totali in df_arco : {len(df_arco)}")
df_arco[["item", "label", "subject"]].head(10)

Righe totali in df_arco : 1120195


,item,label,subject
0,https://w3id.org/arco/resource/HistoricOrArtisticProperty/0500704280,"Arlecchino primordiale. Fine del cinquecento. Martinelli, figurino per costume di scena, maschile (disegno) by Fo, Dario (XX)","figurino per costume di scena, maschile"
2,https://w3id.org/arco/resource/HistoricOrArtisticProperty/0500704131,"Pulcinella, maschera della Commedia dell'Arte (maschera) di Sartori, Donato (XX)",maschera della Commedia dell'Arte
4,https://w3id.org/arco/resource/HistoricOrArtisticProperty/0500704143,"Truffaldino, maschera della Commedia dell'Arte (maschera) di Sartori, Donato (XX)",maschera della Commedia dell'Arte
6,https://w3id.org/arco/resource/HistoricOrArtisticProperty/0500704560,"studio per maschera d'oro (dei re del Gipango) (disegno) by Fo, Dario (XX)",studio per maschera d'oro (dei re del Gipango)
8,https://w3id.org/arco/resource/HistoricOrArtisticProperty/0500704857,"Studio per Medea, figura femminile a mezzo busto (disegno) di Fo, Dario (XX)",figura femminile a mezzo busto
10,https://w3id.org/arco/resource/Lombardia/HistoricOrArtisticProperty/0302126313,Grappolo d'uva (tazzina da caffè) by Ceramiche Pareschi (ultimo quarto sec. XX),Grappolo d'uva
12,https://w3id.org/arco/resource/HistoricOrArtisticProperty/0500704652,"uomo seduto con tavola, arcangelo e figure maschili, studio per manifesto (disegno) by Fo, Dario (XX)",studio per manifesto
16,https://w3id.org/arco/resource/HistoricOrArtisticProperty/0500704575,"Regina Isabella, figurino per costume si scena, femminile, in stile cinquecentesco (disegno) by Fo, Dario, Carbonell, Miguel (XX)","figurino per costume si scena, femminile, in stile cinquecentesco"
18,https://w3id.org/arco/resource/HistoricOrArtisticProperty/0500704163,"Arlecchino primordiale, figurino per costume di scena, maschile (disegno) by Fo, Dario (XX)","figurino per costume di scena, maschile"
20,https://w3id.org/arco/resource/Lombardia/HistoricOrArtisticProperty/0302021552,"Kappa su granchio, CREATURA FANTASTICA e ANIMALE (scultura) di Ikkosai Toun (metà sec. XIX)",CREATURA FANTASTICA e ANIMALE


In [275]:
import re

# 1. filtro base (toglie null)
df_arco = df_arco[df_arco["subject"].notna()].copy()

#termini da escludere (non mitologici)
ignore_subjects = [
    "Acheo","Africa","Agatone","Aia","Aix","Alessandro","Alessi","Angelo","Aniceto","Anna","Antenore","Antioco","Api","Arche","Archelao","Armeno",
    "Astarte","Asteria","Asterio","Asia","Argia","Astaco","Atteo","Aula","Aventino","Basile","Bolina","Bronte","Calice","Camilla", "Canto","Capeto","Car",
    "Cassandra","Cerano", "Cino","Cleopatra","Cocito","Como", "Copia","Corinto","Coro","Creta", "Damaso","Delfino", "Egisto","Egitto","Egle","Elea", "Eleuteria", "Eleutero", "Elia",
    "Elios","Emo", "Epulone","Erme","Eschilo","Esione","Etna", "Etolo"
]


df_arco = df_arco[
    ~df_arco["subject"].str.contains(
        "|".join(ignore_subjects),
        case=False,
        na=False
    )
]


terms = [
    re.escape(t)
    for t in myth_terms
    if isinstance(t, str)
    and t.strip() != ""
    and t[0].isupper()
]

# regex unica case-sensitive (matching esatto)
pattern = r'(?<![A-Za-zÀ-ÿ])(' + '|'.join(terms) + r')(?![A-Za-zÀ-ÿ-])'
regex = re.compile(pattern)


#filtro per regole contestuali termini ambigui
context_rules = {
    "Ascanio" : ["fuoco", "cervo"],
    "Callisto": ["Arcade", "Diana"],
    "Concordia": ["Allegoria", "Innocenza", "Giustizia"],
    "Cornacchia": ["metamorfosi"],
    "Egeo": ["Teseo"],
    "Elena": ["ratto"],
    "Ettore": ["morte", "Protesilao", "Achille", "Andromaca", "Paride"],
}


def validate_matches(row):

    subject = row["subject"]

    valid_matches = []

    for match in row["matched_term"]:

        # se il termine non ha regole → tienilo
        if match not in context_rules:
            valid_matches.append(match)
            continue

        # controlla parole contestuali con word boundaries (parole intere)
        keywords = context_rules[match]

        if any(re.search(r'\b' + re.escape(k) + r'\b', subject, re.IGNORECASE) for k in keywords):
            valid_matches.append(match)

    return valid_matches


df_arco["matched_term"] = df_arco["subject"].str.findall(regex)
# matching veloce
df_arco["matched_term"] = df_arco.apply(validate_matches, axis=1)

# solo match
df_matches = df_arco[df_arco["matched_term"].str.len() > 0]


print(f"Righe con almeno 1 match: {len(df_matches)}")

Righe con almeno 1 match: 12475


In [279]:
# Verifica i risultati
print(f"Totale beni Arco: {len(df_arco)}")
print(f"Beni con match mitologico: {len(df_matches)}")

Totale beni Arco: 471804
Beni con match mitologico: 12475


In [289]:
pd.set_option("display.max_colwidth", None)

start = 400
end = 600


df_matches_sorted = df_matches.sort_values("matched_term")

df_matches_sorted[["item","label", "subject", "matched_term"]].iloc[start:end]


,item,label,subject,matched_term
361050,https://w3id.org/arco/resource/HistoricOrArtisticProperty/0900383414-7,"episodio della vita di Anassarete (stampa smarginata, serie) by Le Pautre Jean (seconda metà sec. XVII)",episodio della vita di Anassarete,[Anassarete]
805780,https://w3id.org/arco/resource/HistoricOrArtisticProperty/0900478607,"celebrazione dell'anniversario della morte di Anchise (stampa, elemento d'insieme) by Pinelli Bartolomeo (sec. XIX)",celebrazione dell'anniversario della morte di Anchise,[Anchise]
743492,https://w3id.org/arco/resource/HistoricOrArtisticProperty/1600211214,Anchise comanda di partire da Troia (stampa) - ambito italiano (sec. XVII),Anchise comanda di partire da Troia,[Anchise]
459866,https://w3id.org/arco/resource/HistoricOrArtisticProperty/0700034272-3,"Andromaca desolata per la perdita del marito (dipinto, elemento d'insieme) di Frascheri Giuseppe (metà sec. XIX)",Andromaca desolata per la perdita del marito,[Andromaca]
1953504,https://w3id.org/arco/resource/HistoricOrArtisticProperty/0300150370,"Andromaca piange Ettore morto (stampa, elemento d'insieme) di Cunego Domenico, Hamilton Gavin (sec. XVIII)",Andromaca piange Ettore morto,"[Andromaca, Ettore]"
1158888,https://w3id.org/arco/resource/HistoricOrArtisticProperty/0800034148,Elena; Andromaca; Ettore; Achille (rilievo) - bottega romagnola (seconda metà sec. XVIII),Elena; Andromaca; Ettore; Achille,"[Andromaca, Ettore, Achille]"
189308,https://w3id.org/arco/resource/HistoricOrArtisticProperty/0800083476,"Andromeda incatenata (stampa) by Furini Francesco, Pacini Sante, Schweickart Johann Adam (sec. XVIII)",Andromeda incatenata,[Andromeda]
1707248,https://w3id.org/arco/resource/HistoricOrArtisticProperty/0500187409,Liberazione di Andromeda con motivi decorativi geometrici e vegetali (tazza) by Manifattura Imperiale di Vienna (XIX),Liberazione di Andromeda con motivi decorativi geometrici e vegetali,[Andromeda]
963484,https://w3id.org/arco/resource/HistoricOrArtisticProperty/1201007543,Andromeda (dipinto) di Furini Francesco (secondo quarto sec. XVII),Andromeda,[Andromeda]
804788,https://w3id.org/arco/resource/HistoricOrArtisticProperty/0900457727,Andromeda incatenata (dipinto) - ambito fiorentino (terzo quarto sec. XVII),Andromeda incatenata,[Andromeda]


In [284]:
# codice per serializzare dataframe in triple (aggiungere più campi al dataframe con query sparql a partire dal dataframe filtrato sul soggetto)
df_matched_items = df_matches[["item"]]
# transform item df in a list of unique values
arco_items_list = list(set(df_matched_items))

# extend_matched_items
def extend_matched_items(items_list):
    # define a batch size
    batch_size = 500
    all_dfs = []

    for i in range(0, len(items_list), batch_size):
        # slice the list
        batch = items_list[i:i + batch_size]

        # wrap arco URIs in <>
        formatted_items = " ".join([f"<{uri}>" for uri in batch])

        query_myth_items = """
        SELECT ?identifier 
            (GROUP_CONCAT(DISTINCT ?creator; separator=", ") AS ?creators) 
            (GROUP_CONCAT(DISTINCT ?type; separator=", ") AS ?types) 
            (GROUP_CONCAT(DISTINCT ?materialOrTechnique; separator=", ") AS ?materialsOrTechniques) 
            (GROUP_CONCAT(DISTINCT ?instituteOrSite; separator=", ") AS ?institutesOrSites) 
            (GROUP_CONCAT(DISTINCT ?creationLocation; separator=", ") AS ?creationLocations)
        WHERE {
            VALUES ?item {"""+formatted_items+"""} .
            ?item arco:uniqueIdentifier ?identifier .

            OPTIONAL {?item dc:creator ?creator . }
            OPTIONAL {?item a-dd:hasCulturalPropertyType ?type . } # per estrarre tipologia opera (es: dipinto)
            OPTIONAL {?item a-dd:hasMaterialOrTechnique ?materialOrTechnique . }
            OPTIONAL {?item a-loc:hasCulturalInstituteOrSite ?instituteOrSite . }
            OPTIONAL {?item a-cd:hasCreationLocation ?creationLocation . } # per estrarre luogo di conservazione
            # non l'ho ancora fatto ma possiamo aggiungere anche un'immagine con foaf:depiction
                
        }

        GROUP BY ?identifier
        """
        sparql_arco.setQuery(query_myth_items)
        sparql_arco.setReturnFormat(JSON)
        try:
            query_results = sparql_arco.query().convert()
            query_data = []
            
            # Iteriamo sui risultati della query corrente
            for result in query_results["results"]["bindings"]:
                query_data.append({
                    "identifier": result["identifier"]["value"],
                    "creator": result.get("creators", {}).get("value"),
                    "type": result.get("types", {}).get("value"),
                    "materialOrTechnique": result.get("materialsOrTechniques", {}).get("value"),
                    "instituteOrSite": result.get("institutesOrSites", {}).get("value"),
                    "creationLocation": result.get("creationLocations", {}).get("value")
                })
            
            if query_data:
                # Creiamo un DF per questo batch e lo aggiungiamo alla lista
                all_dfs.append(pd.DataFrame(query_data))
                
        except Exception as e:
            print(f"Errore nel batch {i}: {e}")
    
    # Uniamo tutto alla fine in un unico DataFrame
    if all_dfs:
        return pd.concat(all_dfs, ignore_index=True)
    return pd.DataFrame()


# function call: get the dataframe with all the additional information for each arco item that has mythological subject
additional_info_df = extend_matched_items(arco_items_list)
# drop duplicates identifiers
additional_info_df = additional_info_df.drop_duplicates(subset=['identifier'])

# extend the original dataframe with the additional information using the identifier for merging
# create identifier column in original dataframe
df_matches["identifier"] = df_matches["item"].apply(lambda x: str(x).split('/')[-1])

# merge dataframes on identifier column
df_finale = pd.merge(df_matches, additional_info_df, on='identifier', how='left')

C:\Users\annap\AppData\Local\Temp\ipykernel_16520\622370406.py:78: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_matches["identifier"] = df_matches["item"].apply(lambda x: str(x).split('/')[-1])


KeyError: 'identifier'